# Machine Learning Intro

Assignment answers  |  Source: ml1.pdf

All questions are answered in slightly expanded, student-friendly language. Code blocks are runnable examples; dataset-dependent outputs should be generated locally.

## Question 1. Explain the differences between AI, ML, Deep Learning (DL), and Data Science (DS).

Answer: Artificial Intelligence (AI) is the broad field of creating systems that perform tasks requiring intelligence. Machine Learning (ML) is a subset of AI that learns patterns from data. Deep Learning (DL) is a subset of ML using multilayer neural networks, especially useful for images and language. Data Science (DS) combines statistics, programming and domain knowledge to extract insights; it may use ML but is not limited to it. Example: a sales dashboard is data science, while a model predicting sales is machine learning.

## Question 2. What are the types of machine learning? Describe each with one real-world example.

Answer: Supervised learning uses labeled examples to predict outputs, such as classifying spam emails. Unsupervised learning discovers structure in unlabeled data, such as grouping customers by purchase behavior. Reinforcement learning learns actions from rewards and penalties, such as an agent learning to navigate a game. Semi-supervised learning combines a small labeled set with many unlabeled examples, often for image classification.

## Question 3. Define overfitting, underfitting, and the bias-variance tradeoff in machine learning.

Answer: Overfitting occurs when a model learns training noise and performs poorly on unseen data; it usually has low training error and high test error. Underfitting occurs when a model is too simple to capture patterns, so both errors remain high. Bias represents error from overly restrictive assumptions, while variance represents sensitivity to training data. Cross-validation, regularization and suitable model complexity help balance the two.

## Question 4. What are outliers in a dataset, and list three common techniques for handling them.

Answer: Outliers are observations unusually distant from most values, possibly caused by entry errors or genuine rare events. Three techniques are (1) remove verified erroneous observations, (2) cap extreme values using percentile or IQR thresholds, and (3) transform skewed features, such as using a logarithm. Always investigate outliers before modifying them, especially in fraud detection where rare cases may be important.

## Question 5. Explain the process of handling missing values and mention one imputation technique for numerical and one for categorical data.

Answer: First identify missing values and examine whether missingness is systematic. Split the data before learning imputation values to avoid leakage. For numerical features, median imputation is robust to extreme values; for categorical features, mode imputation replaces missing entries with the most common category. Add a missingness indicator when absence itself may carry useful information.

## Question 6. Write a Python program that: ●​ Creates a synthetic imbalanced dataset with make_classification() from sklearn.datasets. ●​ Prints the class distribution.

Answer: The code generates 1,000 samples with a 90:10 class imbalance and counts each label. The random seed makes the output reproducible.

In [ ]:
from sklearn.datasets import make_classification
import numpy as np
X, y = make_classification(n_samples=1000, n_features=10,
    n_informative=5, weights=[0.9, 0.1], random_state=42)
labels, counts = np.unique(y, return_counts=True)
print(dict(zip(labels, counts)))

## Question 7. Implement one-hot encoding using pandas for the following list of colors: ['Red', 'Green', 'Blue', 'Green', 'Red']. Print the resulting dataframe.​

Answer: One-hot encoding creates one binary column per color, avoiding a misleading numerical order between categories.

In [ ]:
import pandas as pd
colors = ['Red', 'Green', 'Blue', 'Green', 'Red']
df = pd.get_dummies(pd.Series(colors, name='color'), dtype=int)
print(df.to_string(index=False))

## Question 8. Write a Python script to: ●​ Generate 1000 samples from a normal distribution. ●​ Introduce 50 random missing values. ●​ Fill missing values with the column mean. ●​ Plot a histogram before and after imputation.

Answer: The histogram before imputation excludes missing values; the histogram afterward includes the 50 replacements at the observed mean, which may create a spike around the center.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
values = rng.normal(size=1000)
missing_idx = rng.choice(1000, 50, replace=False)
with_missing = values.copy()
with_missing[missing_idx] = np.nan
filled = np.where(np.isnan(with_missing),
                  np.nanmean(with_missing), with_missing)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(with_missing[~np.isnan(with_missing)], bins=25)
axes[0].set_title('Before imputation')
axes[1].hist(filled, bins=25)
axes[1].set_title('After mean imputation')
plt.tight_layout(); plt.show()
print('Missing before:', np.isnan(with_missing).sum())
print('Missing after:', np.isnan(filled).sum())

## Question 9. Implement Min-Max scaling on the following list of numbers [2, 5, 10, 15, 20] using sklearn.preprocessing.MinMaxScaler. Print the scaled array.

Answer: Min-Max scaling transforms the minimum value to 0 and maximum to 1, preserving the relative positions of the intermediate values.

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
x = np.array([2, 5, 10, 15, 20]).reshape(-1, 1)
print(MinMaxScaler().fit_transform(x).ravel())
# [0.         0.16666667 0.44444444 0.72222222 1.        ]

## Question 10. You are working as a data scientist for a retail company. You receive a customer transaction dataset that contains: ●​ Missing ages, ●​ Outliers in transaction amount, ●​ A highly imbalanced target (fraud vs. non-fraud), ●​ Categorical variables like payment method. Explain the step-by-step data preparation plan you’d follow before training a machine learning model. Include how you’d address missing data, outliers, imbalance, and encoding.

Answer: First split the transaction data into training and test sets using stratification, and preserve the test set untouched. Impute missing ages with the training median, cap only implausible transaction amounts after investigation, and one-hot encode payment methods. Use class weights or resampling on training folds only for fraud imbalance. Fit preprocessing in a pipeline, tune with stratified cross-validation, and evaluate precision, recall, F1 and PR-AUC rather than accuracy alone.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
# df must contain age, amount, payment_method and fraud
X = df[['age', 'amount', 'payment_method']]
y = df['fraud']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=.2, random_state=42)
pre = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), ['age','amount']),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('encode', OneHotEncoder(handle_unknown='ignore'))]),
     ['payment_method'])])
model = Pipeline([('preprocess', pre),
                  ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000))])
model.fit(X_train, y_train)
print('Test score:', model.score(X_test, y_test))
# Replace accuracy with a full fraud-focused evaluation.